In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union
import re
import pandas as pd


# =========================
# Exceptions
# =========================

class ParseError(Exception):
    """Raised when DSL cannot be parsed into an AST."""
    pass


class EvalError(Exception):
    """Raised when AST cannot be executed on a given table (type/arity/unknown op/etc)."""
    pass


# =========================
# AST nodes
# =========================

@dataclass(frozen=True)
class Literal:
    raw: str


@dataclass(frozen=True)
class Call:
    name: str
    args: List[Any]  # List[Literal|Call]


@dataclass(frozen=True)
class Program:
    expr: Any        # Literal|Call
    expected: bool   # suffix bool from "=True/False"


# =========================
# Runtime value types
# =========================

Scalar = Union[str, float, bool]
Row = int


@dataclass(frozen=True)
class RowSet:
    rows: List[int]
    hint_field: Optional[str] = None

    def is_empty(self) -> bool:
        return len(self.rows) == 0

    def size(self) -> int:
        return len(self.rows)

    def top(self) -> int:
        if not self.rows:
            raise EvalError("top() on empty RowSet")
        return self.rows[0]

    def bottom(self) -> int:
        if not self.rows:
            raise EvalError("bottom() on empty RowSet")
        return self.rows[-1]


@dataclass(frozen=True)
class ScalarList:
    values: List[Scalar]

    def is_empty(self) -> bool:
        return len(self.values) == 0


@dataclass(frozen=True)
class ExecResult:
    """
    - final: результат всей программы (учитывая суффикс "=True/False"), или None если не исполнилось
    - expr_value: булево значение выражения (до сравнения с суффиксом), или None
    - expected: суффикс, или None
    - error: строка с причиной, если не исполнилось
    - executable: флаг (final != None)
    """
    final: Optional[bool]
    expr_value: Optional[bool]
    expected: Optional[bool]
    error: Optional[str]
    executable: bool


# =========================
# Function registry
# =========================

@dataclass(frozen=True)
class FunctionSpec:
    canonical: str
    aliases: Tuple[str, ...]
    min_args: int
    max_args: Optional[int]  # None => no upper bound
    variadic: bool = False


class FunctionRegistry:
    """
    Реестр языка:
    - приводит алиасы к каноническим именам
    - знает ожидаемую арность (min/max) для arity-repair и проверок
    """

    def __init__(self) -> None:
        self._specs: Dict[str, FunctionSpec] = {}
        self._alias_to_canon: Dict[str, str] = {}
        self._init_specs()

    def _init_specs(self) -> None:
        # Канонические имена и арности взяты из того, что реально встречается в bootstrap_full.json.
        specs = [
            # comparisons
            FunctionSpec("eq", ("eq", "equal"), 2, 2),
            FunctionSpec("not_eq", ("not_eq", "ne", "neq"), 2, 2),
            FunctionSpec("greater", ("greater", "gt", "more_than"), 2, 2),
            FunctionSpec("less", ("less", "lt"), 2, 2),

            # boolean logic
            FunctionSpec("and", ("and",), 2, None, variadic=True),
            FunctionSpec("or", ("or",), 2, None, variadic=True),
            FunctionSpec("not", ("not",), 1, 1),

            # rowset filters
            FunctionSpec("filter_eq", ("filter_eq", "filter_equal"), 3, 3),
            FunctionSpec("filter_not_eq", ("filter_not_eq", "filter_ne"), 3, 3),
            FunctionSpec("filter_greater", ("filter_greater",), 3, 3),
            FunctionSpec("filter_greater_eq", ("filter_greater_eq",), 3, 3),
            FunctionSpec("filter_less", ("filter_less",), 3, 3),
            FunctionSpec("filter_less_eq", ("filter_less_eq",), 3, 3),

            # row navigation / selection
            FunctionSpec("hop", ("hop",), 2, 2),
            FunctionSpec("argmax", ("argmax",), 1, 2),  # редкая 1-арг форма встречается
            FunctionSpec("argmin", ("argmin",), 1, 2),
            FunctionSpec("top", ("top",), 1, 1),
            FunctionSpec("bottom", ("bottom",), 1, 1),

            # aggregates (обычно 2-арг: RowSet; Field)
            FunctionSpec("count", ("count",), 1, 1),
            FunctionSpec("sum", ("sum",), 1, 2),
            FunctionSpec("avg", ("avg",), 1, 2),
            FunctionSpec("max", ("max",), 2, 2),
            FunctionSpec("min", ("min",), 2, 2),
            FunctionSpec("uniq", ("uniq",), 2, 2),
            FunctionSpec("most_freq", ("most_freq",), 2, 2),
            FunctionSpec("half", ("half",), 1, 1),

            # quantifiers / membership
            FunctionSpec("within", ("within",), 3, 3),
            FunctionSpec("not_within", ("not_within",), 3, 3),
            FunctionSpec("any_eq", ("any_eq",), 3, 3),
            FunctionSpec("any", ("any",), 1, 1),
            FunctionSpec("none", ("none",), 1, 1),
            FunctionSpec("only", ("only",), 1, 1),
            FunctionSpec("zero", ("zero",), 1, 1),

            # all_* (обычно 3-арг: RowSet; Field; Value, но редкие 2-арг тоже есть)
            FunctionSpec("all_eq", ("all_eq",), 2, 3),
            FunctionSpec("all_not_eq", ("all_not_eq",), 3, 3),
            FunctionSpec("all_greater", ("all_greater",), 2, 3),
            FunctionSpec("all_greater_eq", ("all_greater_eq",), 3, 3),
            FunctionSpec("all_less", ("all_less",), 3, 3),
            FunctionSpec("all_less_eq", ("all_less_eq",), 3, 3),

            # arithmetic
            FunctionSpec("diff", ("diff",), 2, 2),
            FunctionSpec("add", ("add",), 2, 2),

            # order / positional predicates (table order)
            FunctionSpec("before", ("before",), 2, 2),
            FunctionSpec("after", ("after",), 2, 2),
            FunctionSpec("first", ("first",), 2, 2),
            FunctionSpec("second", ("second",), 2, 2),
            FunctionSpec("third", ("third",), 2, 2),
            FunctionSpec("fourth", ("fourth",), 2, 2),
            FunctionSpec("fifth", ("fifth",), 2, 2),
            FunctionSpec("last", ("last",), 2, 2),

            # rare
            FunctionSpec("rank", ("rank",), 2, 2),
        ]

        for spec in specs:
            self._specs[spec.canonical] = spec
            for a in spec.aliases:
                self._alias_to_canon[a] = spec.canonical

    def canonicalize(self, name: str) -> str:
        n = name.strip()
        if n in self._alias_to_canon:
            return self._alias_to_canon[n]
        # если внезапно встретится новая функция — пусть падает как unknown
        return n

    def get_spec(self, canonical: str) -> Optional[FunctionSpec]:
        return self._specs.get(canonical)

    def arity_bounds(self, canonical: str) -> Tuple[int, Optional[int], bool]:
        spec = self.get_spec(canonical)
        if spec is None:
            raise EvalError(f"Unknown function: {canonical}")
        return spec.min_args, spec.max_args, spec.variadic


# =========================
# Number parsing (smart-number)
# =========================

class NumberParser:
    """
    Делает из строки число, если это разумно.
    Поддерживает:
    - M:SS(.xx) -> секунды
    - mixed fraction: a - b / c
    - fraction: b / c
    - fallback: первое число в строке ("89.7 fm" -> 89.7)
    """

    _re_time = re.compile(r"^\s*(\d+)\s*:\s*(\d+(?:\.\d+)?)\s*$")
    _re_mixed = re.compile(r"^\s*(\d+)\s*-\s*(\d+)\s*/\s*(\d+)\s*$")
    _re_frac = re.compile(r"^\s*(\d+)\s*/\s*(\d+)\s*$")
    _re_first_num = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

    def parse_literal(self, s: str) -> Optional[float]:
        if s is None:
            return None
        t = str(s).strip()
        if not t:
            return None

        # remove thousands separators
        t2 = t.replace(",", "")

        m = self._re_time.match(t2)
        if m:
            mm = float(m.group(1))
            ss = float(m.group(2))
            return mm * 60.0 + ss

        m = self._re_mixed.match(t2)
        if m:
            a = float(m.group(1))
            b = float(m.group(2))
            c = float(m.group(3))
            if c == 0:
                return None
            return a + (b / c)

        m = self._re_frac.match(t2)
        if m:
            b = float(m.group(1))
            c = float(m.group(2))
            if c == 0:
                return None
            return b / c

        # strict float?
        try:
            return float(t2)
        except Exception:
            pass

        # fallback: first number in string
        m = self._re_first_num.search(t2)
        if m:
            try:
                return float(m.group(0))
            except Exception:
                return None
        return None


# =========================
# Table context
# =========================

class TableContext:
    """
    Обёртка над DataFrame:
    - гарантирует индекс 0..n-1
    - резолвит колонки case-insensitive
    - нормализует текст и извлекает числа
    """

    def __init__(self, df: pd.DataFrame, num_parser: NumberParser) -> None:
        self.df = df.reset_index(drop=True)
        self.num_parser = num_parser
        self.col_map = self._build_col_map(self.df)

    def _build_col_map(self, df: pd.DataFrame) -> Dict[str, str]:
        m: Dict[str, str] = {}
        for c in df.columns:
            m[str(c).strip().lower()] = c
        return m

    def resolve_col(self, field_name: str) -> str:
        key = str(field_name).strip().lower()
        if key in self.col_map:
            return self.col_map[key]
        raise EvalError(f"Unknown column: {field_name}")

    def cell(self, row: int, field_name: str) -> str:
        col = self.resolve_col(field_name)
        v = self.df.at[row, col]
        if v is None:
            return ""
        return str(v)

    def norm_text(self, s: Any) -> str:
        t = "" if s is None else str(s)
        t = t.strip()

        # Преобразуем HTML-кавычки в обычные кавычки
        t = t.replace("&#34;", '"').replace("&quot;", '"')

        # Уберём внешние кавычки, если они реально обрамляют строку
        if len(t) >= 2 and t[0] == '"' and t[-1] == '"':
            t = t[1:-1].strip()

        return t.strip().lower()

    def to_number(self, s: Any) -> Optional[float]:
        # bool не превращаем в числа
        if isinstance(s, bool):
            return None
        if isinstance(s, (int, float)):
            return float(s)
        return self.num_parser.parse_literal(str(s))

    def cmp_eq(self, a: Any, b: Any) -> bool:
        na = self.to_number(a)
        nb = self.to_number(b)
        if na is not None and nb is not None:
            return na == nb
        return self.norm_text(a) == self.norm_text(b)

    def cmp_not_eq(self, a: Any, b: Any) -> bool:
        return not self.cmp_eq(a, b)

    def cmp_greater(self, a: Any, b: Any) -> bool:
        na = self.to_number(a)
        nb = self.to_number(b)
        if na is not None and nb is not None:
            return na > nb
        # если чисел нет — лучше не делать лексикографию, а считать невалидным
        raise EvalError("greater requires numeric operands")

    def cmp_less(self, a: Any, b: Any) -> bool:
        na = self.to_number(a)
        nb = self.to_number(b)
        if na is not None and nb is not None:
            return na < nb
        raise EvalError("less requires numeric operands")

    def cmp_greater_eq(self, a: Any, b: Any) -> bool:
        na = self.to_number(a)
        nb = self.to_number(b)
        if na is not None and nb is not None:
            return na >= nb
        raise EvalError("greater_eq requires numeric operands")

    def cmp_less_eq(self, a: Any, b: Any) -> bool:
        na = self.to_number(a)
        nb = self.to_number(b)
        if na is not None and nb is not None:
            return na <= nb
        raise EvalError("less_eq requires numeric operands")


# =========================
# Parser (Program -> AST)
# =========================

class ProgramParser:
    """
    Парсер DSL:
    - quote-aware + brace-depth-aware
    - поддерживает разделители ';' и ','
    - делает arity-repair под реальные шумы из bootstrap_full.json
    """

    _re_ident = re.compile(r"^([A-Za-z_][A-Za-z0-9_]*)\{")

    def __init__(self, registry: FunctionRegistry) -> None:
        self.registry = registry
        self._cache: Dict[str, Program] = {}

    def parse_program(self, text: str) -> Program:
        key = text
        if key in self._cache:
            return self._cache[key]

        s = self._preprocess(text.strip())
        eq_pos = self._find_last_top_level_equals(s)
        if eq_pos is None:
            raise ParseError("Program must end with =True or =False")

        expr_str = s[:eq_pos].strip()
        suffix = s[eq_pos + 1:].strip().lower()
        if suffix not in ("true", "false"):
            raise ParseError(f"Bad suffix: {suffix}")

        prog = Program(expr=self.parse_expr(expr_str), expected=(suffix == "true"))
        self._cache[key] = prog
        return prog

    def parse_expr(self, text: str) -> Any:
        t = self._preprocess(text.strip())
        m = self._re_ident.match(t)
        if not m:
            return Literal(t)

        name = m.group(1)
        name_c = self.registry.canonicalize(name)

        # позиция открывающей '{' сразу после имени
        open_pos = len(name)
        if open_pos >= len(t) or t[open_pos] != "{":
            return Literal(t)

        close_pos = self._find_matching_brace(t, open_pos)
        if close_pos is None or close_pos != len(t) - 1:
            # либо не закрылась, либо есть хвост -> считаем литералом (консервативно)
            return Literal(t)

        inside = t[open_pos + 1: close_pos]
        raw_args = self._split_args(inside)
        repaired = self._repair_args(name_c, raw_args)

        args_ast = [self.parse_expr(a) for a in repaired]
        return Call(name=name_c, args=args_ast)

    # ---------- preprocessing & scanning ----------

    def _preprocess(self, s: str) -> str:
        # Важно: превращаем HTML-кавычки в обычные, чтобы splitter мог игнорировать delimiters внутри quotes
        return s.replace("&#34;", '"').replace("&quot;", '"')

    def _find_last_top_level_equals(self, s: str) -> Optional[int]:
        depth = 0
        in_quote = False
        pos = None
        for i, ch in enumerate(s):
            if ch == '"':
                in_quote = not in_quote
                continue
            if in_quote:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
            elif ch == "=" and depth == 0:
                pos = i
        return pos

    def _find_matching_brace(self, s: str, open_pos: int) -> Optional[int]:
        depth = 0
        in_quote = False
        for i in range(open_pos, len(s)):
            ch = s[i]
            if ch == '"':
                in_quote = not in_quote
                continue
            if in_quote:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return i
        return None

    def _has_top_level_semicolon(self, s: str) -> bool:
        depth = 0
        in_quote = False
        for ch in s:
            if ch == '"':
                in_quote = not in_quote
                continue
            if in_quote:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
            elif ch == ";" and depth == 0:
                return True
        return False

    def _split_args(self, inside: str) -> List[str]:
        """
        Делит аргументы внутри {...} по верхнеуровневому разделителю:
        - если есть ';' на depth=0 => делим по ';'
        - иначе делим по ','
        quote-aware и brace-aware.
        Важно: НЕ выкидываем пустые аргументы.
        """
        delim = ";" if self._has_top_level_semicolon(inside) else ","

        parts: List[str] = []
        buf: List[str] = []
        depth = 0
        in_quote = False

        for ch in inside:
            if ch == '"':
                in_quote = not in_quote
                buf.append(ch)
                continue

            if not in_quote:
                if ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1

                if ch == delim and depth == 0:
                    parts.append("".join(buf).strip())
                    buf = []
                    continue

            buf.append(ch)

        parts.append("".join(buf).strip())
        return parts

    # ---------- arity repair ----------

    def _repair_args(self, canonical_name: str, raw_args: List[str]) -> List[str]:
        """
        Чинит реальные шумы:
        - filter_* ожидает 3 args, но приходит 2: "field, value" в одном аргументе -> делим
        - если аргументов слишком много (обычно из-за запятых) -> склеиваем хвост в последний
        - unary ops с лишними аргументами -> отбрасываем лишние (count/only/etc)
        """
        spec = self.registry.get_spec(canonical_name)
        if spec is None:
            return raw_args

        args = list(raw_args)

        # 1) специальные кейсы: filter_* с 2 args вместо 3 (частный шум)
        if canonical_name in ("filter_less", "filter_eq", "filter_not_eq",
                              "filter_greater", "filter_greater_eq", "filter_less_eq") and len(args) == 2:
            # pattern: filter_less{C; field, value}
            split = self._split_field_value_pair(args[1])
            if split is not None:
                args = [args[0], split[0], split[1]]

        # 2) редкий шум: filter_eq может прийти с 4 аргументами из-за запятых в value
        #    приводим к 3, склеивая tail
        if canonical_name.startswith("filter_") and spec.max_args == 3 and len(args) > 3:
            args = [args[0], args[1], self._join_tail(args[2:])]

        # 3) сравнения бывают "not_eq" с 3 аргументами из-за запятых -> склеить в 2
        if canonical_name in ("eq", "not_eq", "greater", "less") and len(args) > 2:
            args = [args[0], self._join_tail(args[1:])]

        # 4) unary ops с лишними аргументами (в local edits встречается)
        if canonical_name in ("count", "only", "top", "bottom", "half", "any", "none", "zero", "not") and len(args) > 1:
            args = [args[0]]

        # 5) add/diff строго бинарные: если вдруг 3 -> склеим в 2
        if canonical_name in ("add", "diff") and len(args) > 2:
            args = [args[0], self._join_tail(args[1:])]

        # 6) all_eq / all_greater допускают 2-арг вариант (редко) и 3-арг (обычно).
        #    если >3 -> склеить хвост в последний
        if canonical_name in ("all_eq", "all_greater") and len(args) > 3:
            args = [args[0], args[1], self._join_tail(args[2:])]

        # 7) and/or — variadic: просто обрежем пустые хвосты, но не ниже min_args
        #    (пустые аргументы иногда получаются из ";;")
        if canonical_name in ("and", "or"):
            args = [a for a in args if a != ""]
            # если после чистки стало слишком мало — оставим как есть (упадёт на Eval)

        # финально: если max_args задан и всё ещё больше — склеить tail
        if spec.max_args is not None and len(args) > spec.max_args:
            head = args[:spec.max_args - 1]
            tail = args[spec.max_args - 1:]
            args = head + [self._join_tail(tail)]

        return args

    def _split_field_value_pair(self, s: str) -> Optional[Tuple[str, str]]:
        """
        Делит строку "field, value" на (field, value), но только на верхнем уровне и с учётом кавычек.
        Берём последнее вхождение ',' (обычно value может содержать запятые).
        """
        t = s.strip()
        if not t or "," not in t:
            return None

        in_quote = False
        last_comma = None
        for i, ch in enumerate(t):
            if ch == '"':
                in_quote = not in_quote
                continue
            if not in_quote and ch == ",":
                last_comma = i

        if last_comma is None:
            return None

        left = t[:last_comma].strip()
        right = t[last_comma + 1:].strip()
        if not left or not right:
            return None
        return left, right

    def _join_tail(self, parts: List[str]) -> str:
        # Склеиваем через "," — это безопаснее: чаще всего хвост образовался из-за запятых в литералах.
        return ", ".join([p.strip() for p in parts])


# =========================
# Engine (AST execution)
# =========================

class TabFactEngine:
    """
    Исполнитель DSL:
    - parse -> AST -> eval
    - строгие типы (RowSet/ScalarList)
    - операции реализованы как методы (без вложенных функций)
    """

    def __init__(self) -> None:
        self.registry = FunctionRegistry()
        self.parser = ProgramParser(self.registry)
        self.num_parser = NumberParser()

        # dispatch table: canonical_name -> method
        self._ops: Dict[str, Any] = {
            # comparisons
            "eq": self.op_eq,
            "not_eq": self.op_not_eq,
            "greater": self.op_greater,
            "less": self.op_less,

            # boolean logic
            "and": self.op_and,
            "or": self.op_or,
            "not": self.op_not,

            # filters
            "filter_eq": self.op_filter_eq,
            "filter_not_eq": self.op_filter_not_eq,
            "filter_greater": self.op_filter_greater,
            "filter_greater_eq": self.op_filter_greater_eq,
            "filter_less": self.op_filter_less,
            "filter_less_eq": self.op_filter_less_eq,

            # navigation
            "hop": self.op_hop,
            "argmax": self.op_argmax,
            "argmin": self.op_argmin,
            "top": self.op_top,
            "bottom": self.op_bottom,

            # aggregates
            "count": self.op_count,
            "sum": self.op_sum,
            "avg": self.op_avg,
            "max": self.op_max,
            "min": self.op_min,
            "uniq": self.op_uniq,
            "most_freq": self.op_most_freq,
            "half": self.op_half,

            # quantifiers
            "within": self.op_within,
            "not_within": self.op_not_within,
            "any_eq": self.op_within,   # any_eq семантически то же, что within
            "any": self.op_any,
            "none": self.op_none,
            "only": self.op_only,
            "zero": self.op_zero,

            # all_*
            "all_eq": self.op_all_eq,
            "all_not_eq": self.op_all_not_eq,
            "all_greater": self.op_all_greater,
            "all_greater_eq": self.op_all_greater_eq,
            "all_less": self.op_all_less,
            "all_less_eq": self.op_all_less_eq,

            # arithmetic
            "diff": self.op_diff,
            "add": self.op_add,

            # order/positional
            "before": self.op_before,
            "after": self.op_after,
            "first": self.op_first,
            "second": self.op_second,
            "third": self.op_third,
            "fourth": self.op_fourth,
            "fifth": self.op_fifth,
            "last": self.op_last,

            # rare
            "rank": self.op_rank,
        }

    # ---------- public API ----------

    def execute(self, program_text: str, df: pd.DataFrame) -> ExecResult:
        """
        Возвращает ExecResult:
        - если получилось: final != None
        - если нет: final == None и есть error
        """
        try:
            prog = self.parser.parse_program(program_text)
            ctx = TableContext(df, self.num_parser)
            value = self.eval_node(prog.expr, ctx)

            if not isinstance(value, bool):
                raise EvalError(f"Expression did not evaluate to bool, got: {type(value)}")

            final = (value == prog.expected)
            return ExecResult(final=final, expr_value=value, expected=prog.expected, error=None, executable=True)

        except Exception as e:
            return ExecResult(final=None, expr_value=None, expected=None, error=str(e), executable=False)

    # ---------- eval core ----------

    def eval_node(self, node: Any, ctx: TableContext) -> Any:
        if isinstance(node, Literal):
            return self.eval_literal(node, ctx)
        if isinstance(node, Call):
            return self.eval_call(node, ctx)
        raise EvalError(f"Bad AST node: {type(node)}")

    def eval_literal(self, node: Literal, ctx: TableContext) -> Any:
        raw = node.raw.strip()

        # special rowset literal
        if raw == "all_rows":
            return RowSet(rows=list(range(len(ctx.df))), hint_field=None)

        low = raw.lower()
        if low == "true":
            return True
        if low == "false":
            return False

        # keep as string; numbers will be parsed where needed via ctx.to_number(...)
        # but we normalize quotes for literals that are written as "..."
        # (this matters for weird program with "kuala lumpur;")
        if len(raw) >= 2 and raw[0] == '"' and raw[-1] == '"':
            return raw[1:-1]

        return raw

    def eval_call(self, node: Call, ctx: TableContext) -> Any:
        name = self.registry.canonicalize(node.name)
        if name not in self._ops:
            raise EvalError(f"Unknown function: {name}")

        args = [self.eval_node(a, ctx) for a in node.args]
        return self._ops[name](args, ctx)

    # ---------- type helpers ----------

    def as_rowset(self, v: Any) -> RowSet:
        if isinstance(v, RowSet):
            return v
        raise EvalError(f"Expected RowSet, got: {type(v)}")

    def as_row(self, v: Any) -> int:
        if isinstance(v, int):
            return v
        if isinstance(v, RowSet) and v.size() == 1:
            return v.top()
        raise EvalError(f"Expected Row, got: {type(v)}")

    def as_scalar_list(self, v: Any) -> ScalarList:
        if isinstance(v, ScalarList):
            return v
        raise EvalError(f"Expected ScalarList, got: {type(v)}")

    def row_of(self, v: Any) -> int:
        # Для before/after и positional: RowSet -> top row
        if isinstance(v, int):
            return v
        if isinstance(v, RowSet):
            return v.top()
        raise EvalError(f"Expected Row or RowSet, got: {type(v)}")

    # =========================
    # Operations: comparisons
    # =========================

    def op_eq(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("eq", args, 2)
        return ctx.cmp_eq(args[0], args[1])

    def op_not_eq(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("not_eq", args, 2)
        return ctx.cmp_not_eq(args[0], args[1])

    def op_greater(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("greater", args, 2)
        return ctx.cmp_greater(args[0], args[1])

    def op_less(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("less", args, 2)
        return ctx.cmp_less(args[0], args[1])

    # =========================
    # Operations: boolean logic
    # =========================

    def op_and(self, args: List[Any], ctx: TableContext) -> bool:
        if len(args) < 2:
            raise EvalError("and requires at least 2 args")
        return all(bool(a) for a in args)

    def op_or(self, args: List[Any], ctx: TableContext) -> bool:
        if len(args) < 2:
            raise EvalError("or requires at least 2 args")
        return any(bool(a) for a in args)

    def op_not(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("not", args, 1)
        return not bool(args[0])

    # =========================
    # Operations: filters
    # =========================

    def op_filter_eq(self, args: List[Any], ctx: TableContext) -> RowSet:
        self._require_arity("filter_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        value = args[2]
        out = []
        for r in C.rows:
            if ctx.cmp_eq(ctx.cell(r, field), value):
                out.append(r)
        return RowSet(rows=out, hint_field=field)

    def op_filter_not_eq(self, args: List[Any], ctx: TableContext) -> RowSet:
        self._require_arity("filter_not_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        value = args[2]
        out = []
        for r in C.rows:
            if ctx.cmp_not_eq(ctx.cell(r, field), value):
                out.append(r)
        return RowSet(rows=out, hint_field=field)

    def op_filter_greater(self, args: List[Any], ctx: TableContext) -> RowSet:
        self._require_arity("filter_greater", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        value = args[2]
        out = []
        for r in C.rows:
            cell_v = ctx.cell(r, field)
            try:
                if ctx.cmp_greater(cell_v, value):
                    out.append(r)
            except EvalError:
                # если нечисло — просто не проходит фильтр
                continue
        return RowSet(rows=out, hint_field=field)

    def op_filter_greater_eq(self, args: List[Any], ctx: TableContext) -> RowSet:
        self._require_arity("filter_greater_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        value = args[2]
        out = []
        for r in C.rows:
            cell_v = ctx.cell(r, field)
            try:
                if ctx.cmp_greater_eq(cell_v, value):
                    out.append(r)
            except EvalError:
                continue
        return RowSet(rows=out, hint_field=field)

    def op_filter_less(self, args: List[Any], ctx: TableContext) -> RowSet:
        self._require_arity("filter_less", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        value = args[2]
        out = []
        for r in C.rows:
            cell_v = ctx.cell(r, field)
            try:
                if ctx.cmp_less(cell_v, value):
                    out.append(r)
            except EvalError:
                continue
        return RowSet(rows=out, hint_field=field)

    def op_filter_less_eq(self, args: List[Any], ctx: TableContext) -> RowSet:
        self._require_arity("filter_less_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        value = args[2]
        out = []
        for r in C.rows:
            cell_v = ctx.cell(r, field)
            try:
                if ctx.cmp_less_eq(cell_v, value):
                    out.append(r)
            except EvalError:
                continue
        return RowSet(rows=out, hint_field=field)

    # =========================
    # Operations: navigation
    # =========================

    def op_hop(self, args: List[Any], ctx: TableContext) -> Union[Scalar, ScalarList]:
        self._require_arity("hop", args, 2)
        base = args[0]
        field = str(args[1])

        # RowSet -> scalar if size==1, else ScalarList
        if isinstance(base, RowSet):
            if base.size() == 1:
                return ctx.cell(base.top(), field)
            return ScalarList([ctx.cell(r, field) for r in base.rows])

        # Row -> scalar
        if isinstance(base, int):
            return ctx.cell(base, field)

        raise EvalError(f"hop expects Row or RowSet, got: {type(base)}")

    def op_argmax(self, args: List[Any], ctx: TableContext) -> int:
        if len(args) == 1:
            C = self.as_rowset(args[0])
            if C.hint_field is None:
                raise EvalError("argmax{C} requires hint_field (use argmax{C; field})")
            field = C.hint_field
        else:
            self._require_arity("argmax", args, 2)
            C = self.as_rowset(args[0])
            field = str(args[1])

        best_r = None
        best_v = None
        for r in C.rows:
            n = ctx.to_number(ctx.cell(r, field))
            if n is None:
                continue
            if best_r is None or n > best_v:
                best_r = r
                best_v = n
        if best_r is None:
            raise EvalError("argmax: no numeric values")
        return best_r

    def op_argmin(self, args: List[Any], ctx: TableContext) -> int:
        if len(args) == 1:
            C = self.as_rowset(args[0])
            if C.hint_field is None:
                raise EvalError("argmin{C} requires hint_field (use argmin{C; field})")
            field = C.hint_field
        else:
            self._require_arity("argmin", args, 2)
            C = self.as_rowset(args[0])
            field = str(args[1])

        best_r = None
        best_v = None
        for r in C.rows:
            n = ctx.to_number(ctx.cell(r, field))
            if n is None:
                continue
            if best_r is None or n < best_v:
                best_r = r
                best_v = n
        if best_r is None:
            raise EvalError("argmin: no numeric values")
        return best_r

    def op_top(self, args: List[Any], ctx: TableContext) -> int:
        self._require_arity("top", args, 1)
        C = self.as_rowset(args[0])
        return C.top()

    def op_bottom(self, args: List[Any], ctx: TableContext) -> int:
        self._require_arity("bottom", args, 1)
        C = self.as_rowset(args[0])
        return C.bottom()

    # =========================
    # Operations: aggregates
    # =========================

    def op_count(self, args: List[Any], ctx: TableContext) -> float:
        self._require_arity("count", args, 1)
        C = self.as_rowset(args[0])
        return float(C.size())

    def op_sum(self, args: List[Any], ctx: TableContext) -> float:
        if len(args) == 1:
            xs = self._values_from_scalarlist_arg(args[0])
            return float(sum(xs))
        self._require_arity("sum", args, 2)
        xs = self._values_from_rowset_field(args[0], args[1], ctx)
        return float(sum(xs))

    def op_avg(self, args: List[Any], ctx: TableContext) -> float:
        if len(args) == 1:
            xs = self._values_from_scalarlist_arg(args[0])
        else:
            self._require_arity("avg", args, 2)
            xs = self._values_from_rowset_field(args[0], args[1], ctx)

        if not xs:
            raise EvalError("avg: empty/non-numeric")
        return float(sum(xs) / len(xs))

    def op_max(self, args: List[Any], ctx: TableContext) -> float:
        self._require_arity("max", args, 2)
        xs = self._values_from_rowset_field(args[0], args[1], ctx)
        if not xs:
            raise EvalError("max: empty/non-numeric")
        return float(max(xs))

    def op_min(self, args: List[Any], ctx: TableContext) -> float:
        self._require_arity("min", args, 2)
        xs = self._values_from_rowset_field(args[0], args[1], ctx)
        if not xs:
            raise EvalError("min: empty/non-numeric")
        return float(min(xs))

    def op_uniq(self, args: List[Any], ctx: TableContext) -> float:
        self._require_arity("uniq", args, 2)
        C = self.as_rowset(args[0])
        field = str(args[1])
        seen = set()
        for r in C.rows:
            seen.add(ctx.norm_text(ctx.cell(r, field)))
        return float(len(seen))

    def op_most_freq(self, args: List[Any], ctx: TableContext) -> str:
        self._require_arity("most_freq", args, 2)
        C = self.as_rowset(args[0])
        field = str(args[1])
        counts: Dict[str, int] = {}
        for r in C.rows:
            key = ctx.norm_text(ctx.cell(r, field))
            counts[key] = counts.get(key, 0) + 1
        if not counts:
            raise EvalError("most_freq: empty")
        # max by count, tie-break lexicographically for determinism
        best = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
        return best

    def op_half(self, args: List[Any], ctx: TableContext) -> float:
        self._require_arity("half", args, 1)
        C = self.as_rowset(args[0])
        return float(C.size()) / 2.0

    # =========================
    # Operations: quantifiers
    # =========================

    def op_within(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("within", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        value = args[2]
        for r in C.rows:
            if ctx.cmp_eq(ctx.cell(r, field), value):
                return True
        return False

    def op_not_within(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("not_within", args, 3)
        return not self.op_within(args, ctx)

    def op_any(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("any", args, 1)
        v = args[0]
        if isinstance(v, RowSet):
            return not v.is_empty()
        if isinstance(v, ScalarList):
            return not v.is_empty()
        if v is None:
            return False
        if isinstance(v, str):
            return bool(v.strip())
        return True

    def op_none(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("none", args, 1)
        return not self.op_any(args, ctx)

    def op_only(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("only", args, 1)
        C = self.as_rowset(args[0])
        return C.size() == 1

    def op_zero(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("zero", args, 1)
        n = ctx.to_number(args[0])
        if n is None:
            raise EvalError("zero requires numeric operand")
        return n == 0.0

    # =========================
    # Operations: all_*
    # =========================

    def op_all_eq(self, args: List[Any], ctx: TableContext) -> bool:
        if len(args) == 2:
            # редкая форма: all_eq{ScalarList; Value} (в датасете реально есть 3 таких)
            lst = self.as_scalar_list(args[0])
            val = args[1]
            for x in lst.values:
                if not ctx.cmp_eq(x, val):
                    return False
            return True

        self._require_arity("all_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        val = args[2]
        for r in C.rows:
            if not ctx.cmp_eq(ctx.cell(r, field), val):
                return False
        return True

    def op_all_not_eq(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("all_not_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        val = args[2]
        for r in C.rows:
            if ctx.cmp_eq(ctx.cell(r, field), val):
                return False
        return True

    def op_all_greater(self, args: List[Any], ctx: TableContext) -> bool:
        if len(args) == 2:
            # редкая форма: all_greater{ScalarList; Value} (в датасете есть 2 таких)
            lst = self.as_scalar_list(args[0])
            val = args[1]
            for x in lst.values:
                if not ctx.cmp_greater(x, val):
                    return False
            return True

        self._require_arity("all_greater", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        val = args[2]
        for r in C.rows:
            if not ctx.cmp_greater(ctx.cell(r, field), val):
                return False
        return True

    def op_all_greater_eq(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("all_greater_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        val = args[2]
        for r in C.rows:
            if not ctx.cmp_greater_eq(ctx.cell(r, field), val):
                return False
        return True

    def op_all_less(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("all_less", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        val = args[2]
        for r in C.rows:
            if not ctx.cmp_less(ctx.cell(r, field), val):
                return False
        return True

    def op_all_less_eq(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("all_less_eq", args, 3)
        C = self.as_rowset(args[0])
        field = str(args[1])
        val = args[2]
        for r in C.rows:
            if not ctx.cmp_less_eq(ctx.cell(r, field), val):
                return False
        return True

    # =========================
    # Operations: arithmetic
    # =========================

    def op_diff(self, args: List[Any], ctx: TableContext) -> float:
        self._require_arity("diff", args, 2)
        a = ctx.to_number(args[0])
        b = ctx.to_number(args[1])
        if a is None or b is None:
            raise EvalError("diff requires numeric operands")
        return float(a - b)

    def op_add(self, args: List[Any], ctx: TableContext) -> float:
        self._require_arity("add", args, 2)
        a = ctx.to_number(args[0])
        b = ctx.to_number(args[1])
        if a is None or b is None:
            raise EvalError("add requires numeric operands")
        return float(a + b)

    # =========================
    # Operations: order/positional
    # =========================

    def op_before(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("before", args, 2)
        ra = self.row_of(args[0])
        rb = self.row_of(args[1])
        return ra < rb

    def op_after(self, args: List[Any], ctx: TableContext) -> bool:
        self._require_arity("after", args, 2)
        ra = self.row_of(args[0])
        rb = self.row_of(args[1])
        return ra > rb

    def op_first(self, args: List[Any], ctx: TableContext) -> bool:
        return self._positional(args, ctx, pos=1)

    def op_second(self, args: List[Any], ctx: TableContext) -> bool:
        return self._positional(args, ctx, pos=2)

    def op_third(self, args: List[Any], ctx: TableContext) -> bool:
        return self._positional(args, ctx, pos=3)

    def op_fourth(self, args: List[Any], ctx: TableContext) -> bool:
        return self._positional(args, ctx, pos=4)

    def op_fifth(self, args: List[Any], ctx: TableContext) -> bool:
        return self._positional(args, ctx, pos=5)

    def op_last(self, args: List[Any], ctx: TableContext) -> bool:
        # last{C; D}: D находится на последней позиции в C
        self._require_arity("last", args, 2)
        C = self.as_rowset(args[0])
        drow = self.row_of(args[1])
        if C.is_empty():
            return False
        return C.bottom() == drow

    def _positional(self, args: List[Any], ctx: TableContext, pos: int) -> bool:
        self._require_arity("first/second/..", args, 2)
        C = self.as_rowset(args[0])
        drow = self.row_of(args[1])
        idx = pos - 1
        if C.size() <= idx:
            return False
        return C.rows[idx] == drow

    # =========================
    # Operations: rank (rare)
    # =========================

    def op_rank(self, args: List[Any], ctx: TableContext) -> Scalar:
        """
        rank{C; Field}:
        - если есть колонка "rank" в таблице: берём строку с минимальным rank и делаем hop в Field
        - иначе: fallback hop{top{C}; Field}
        """
        self._require_arity("rank", args, 2)
        C = self.as_rowset(args[0])
        out_field = str(args[1])

        # если нет колоноки rank — fallback
        if "rank" not in ctx.col_map:
            r = C.top()
            return ctx.cell(r, out_field)

        best_r = None
        best_v = None
        for r in C.rows:
            n = ctx.to_number(ctx.cell(r, "rank"))
            if n is None:
                continue
            if best_r is None or n < best_v:
                best_r = r
                best_v = n

        if best_r is None:
            best_r = C.top()
        return ctx.cell(best_r, out_field)

    # =========================
    # Helpers: aggregates input
    # =========================

    def _values_from_rowset_field(self, rowset_val: Any, field_val: Any, ctx: TableContext) -> List[float]:
        C = self.as_rowset(rowset_val)
        field = str(field_val)
        xs: List[float] = []
        for r in C.rows:
            n = ctx.to_number(ctx.cell(r, field))
            if n is not None:
                xs.append(n)
        return xs

    def _values_from_scalarlist_arg(self, arg: Any) -> List[float]:
        lst = self.as_scalar_list(arg)
        xs: List[float] = []
        for v in lst.values:
            if isinstance(v, bool):
                continue
            try:
                n = float(str(v).replace(",", ""))
                xs.append(n)
            except Exception:
                continue
        return xs

    # =========================
    # Helpers: arity checks
    # =========================

    def _require_arity(self, name: str, args: List[Any], expected: int) -> None:
        if len(args) != expected:
            raise EvalError(f"{name} expects {expected} args, got {len(args)}")


In [3]:
df = pd.read_csv("/home/chaichuk/frontdoor_llm_causality/statics/datasets/Table-Fact-Checking/data/all_csv/1-2709-4.html.csv", sep="#", dtype=str).fillna("")
engine = TabFactEngine()

res = engine.execute("only{filter_eq{all_rows; format; public radio}}=True", df)
print(res)


ExecResult(final=False, expr_value=False, expected=True, error=None, executable=True)


In [6]:
import unittest
import pandas as pd


class TestParserBasics(unittest.TestCase):
    def setUp(self):
        self.registry = FunctionRegistry()
        self.parser = ProgramParser(self.registry)

    # ---------- helpers ----------

    def assertCall(self, node, name=None, nargs=None):
        self.assertIsInstance(node, Call)
        if name is not None:
            self.assertEqual(node.name, name)
        if nargs is not None:
            self.assertEqual(len(node.args), nargs)

    def assertLit(self, node, raw=None):
        self.assertIsInstance(node, Literal)
        if raw is not None:
            self.assertEqual(node.raw, raw)

    # ---------- suffix parsing ----------

    def test_parse_program_suffix_true(self):
        prog = self.parser.parse_program("eq{1;1}=True")
        self.assertIsInstance(prog, Program)
        self.assertTrue(prog.expected)
        self.assertCall(prog.expr, name="eq", nargs=2)

    def test_parse_program_suffix_false(self):
        prog = self.parser.parse_program("eq{1;1}=False")
        self.assertFalse(prog.expected)
        self.assertCall(prog.expr, name="eq", nargs=2)

    def test_parse_program_missing_suffix_raises(self):
        with self.assertRaises(Exception):
            self.parser.parse_program("eq{1;1}")

    def test_parse_program_bad_suffix_raises(self):
        with self.assertRaises(Exception):
            self.parser.parse_program("eq{1;1}=Maybe")

    def test_last_top_level_equals(self):
        # '=' внутри вложенности не должен считаться суффиксом
        # Суффикс - только последний '=' на top-level
        prog = self.parser.parse_program("eq{diff{2;1};1}=True")
        self.assertTrue(prog.expected)
        self.assertCall(prog.expr, name="eq", nargs=2)
        self.assertCall(prog.expr.args[0], name="diff", nargs=2)

    # ---------- delimiter rules ----------

    def test_split_by_semicolon_when_present(self):
        node = self.parser.parse_expr("eq{a; b}")
        self.assertCall(node, name="eq", nargs=2)
        self.assertLit(node.args[0], "a")
        self.assertLit(node.args[1], "b")

    def test_split_by_comma_when_no_semicolon(self):
        node = self.parser.parse_expr("eq{a, b}")
        self.assertCall(node, name="eq", nargs=2)
        self.assertLit(node.args[0], "a")
        self.assertLit(node.args[1], "b")

    def test_nested_calls_keep_structure(self):
        node = self.parser.parse_expr("and{eq{1;1}; not{false}; or{true; false}}")
        self.assertCall(node, name="and", nargs=3)
        self.assertCall(node.args[0], name="eq", nargs=2)
        self.assertCall(node.args[1], name="not", nargs=1)
        self.assertCall(node.args[2], name="or", nargs=2)

    # ---------- alias canonicalization ----------

    def test_alias_gt_to_greater(self):
        node = self.parser.parse_expr("gt{2;1}")
        self.assertCall(node, name="greater", nargs=2)

    def test_alias_more_than_to_greater(self):
        node = self.parser.parse_expr("more_than{2;1}")
        self.assertCall(node, name="greater", nargs=2)

    def test_alias_equal_to_eq(self):
        node = self.parser.parse_expr("equal{2;1}")
        self.assertCall(node, name="eq", nargs=2)

    def test_alias_neq_to_not_eq(self):
        node = self.parser.parse_expr("neq{2;1}")
        self.assertCall(node, name="not_eq", nargs=2)

    def test_alias_filter_ne_to_filter_not_eq(self):
        node = self.parser.parse_expr("filter_ne{all_rows; col; x}")
        self.assertCall(node, name="filter_not_eq", nargs=3)

    def test_alias_filter_equal_to_filter_eq(self):
        node = self.parser.parse_expr("filter_equal{all_rows; col; x}")
        self.assertCall(node, name="filter_eq", nargs=3)

    # ---------- HTML quotes & quotes handling ----------

    def test_html_quotes_preprocess(self):
        # &#34; -> "  (важно для quote-aware splitting)
        prog = self.parser.parse_program('eq{a; &#34;kuala lumpur;&#34;}=True')
        self.assertCall(prog.expr, name="eq", nargs=2)
        # второй аргумент должен быть одним литералом, без разбиения на ';'
        self.assertLit(prog.expr.args[1], '"kuala lumpur;"')

    def test_semicolon_inside_quotes_not_a_delimiter(self):
        # top-level delim здесь ';', но внутри второго аргумента есть ';' в кавычках
        node = self.parser.parse_expr('eq{a; "kuala lumpur;"}')
        self.assertCall(node, name="eq", nargs=2)
        self.assertLit(node.args[1], '"kuala lumpur;"')

    def test_commas_inside_quotes_not_split_when_semicolon_delim(self):
        node = self.parser.parse_expr('filter_eq{all_rows; name; "a, b, c"}')
        self.assertCall(node, name="filter_eq", nargs=3)
        self.assertLit(node.args[2], '"a, b, c"')

    # ---------- arity repair ----------

    def test_repair_filter_two_args_field_value_in_one(self):
        # filter_less{C; field, value} -> filter_less{C; field; value}
        node = self.parser.parse_expr("filter_less{all_rows; crowd, 15000}")
        self.assertCall(node, name="filter_less", nargs=3)
        self.assertLit(node.args[1], "crowd")
        self.assertLit(node.args[2], "15000")

    def test_repair_unary_extra_args_trim(self):
        node = self.parser.parse_expr("count{all_rows; bogus}")
        self.assertCall(node, name="count", nargs=1)
        self.assertLit(node.args[0], "all_rows")

        node2 = self.parser.parse_expr("only{all_rows; bogus; junk}")
        self.assertCall(node2, name="only", nargs=1)

    def test_repair_comparisons_too_many_args_join_tail(self):
        node = self.parser.parse_expr("eq{a; b; c}")
        self.assertCall(node, name="eq", nargs=2)
        # второй аргумент станет "b, c" из join_tail
        self.assertLit(node.args[1], "b, c")

    def test_repair_filter_too_many_args_join_tail(self):
        node = self.parser.parse_expr("filter_eq{all_rows; name; a; b; c}")
        self.assertCall(node, name="filter_eq", nargs=3)
        self.assertLit(node.args[2], "a, b, c")

    def test_repair_all_eq_more_than_3_join_tail(self):
        node = self.parser.parse_expr("all_eq{all_rows; field; a; b; c}")
        self.assertCall(node, name="all_eq", nargs=3)
        self.assertLit(node.args[2], "a, b, c")

    def test_repair_and_or_drop_empty_args(self):
        node = self.parser.parse_expr("and{true;;false;}")
        self.assertCall(node, name="and")
        # после удаления пустых будет 2 аргумента: true, false
        self.assertEqual(len(node.args), 2)

    # ---------- malformed syntax behavior ----------

    def test_malformed_unbalanced_brace_becomes_literal(self):
        node = self.parser.parse_expr("eq{1;2")
        self.assertLit(node)

    def test_malformed_trailing_garbage_becomes_literal(self):
        node = self.parser.parse_expr("eq{1;2} junk")
        self.assertLit(node)

    def test_unmatched_quote_does_not_crash(self):
        # не гарантируем правильный парсинг, но тестируем устойчивость
        node = self.parser.parse_expr('eq{a; "unterminated}')
        self.assertIsNotNone(node)


class TestEngineExecutionSmoke(unittest.TestCase):
    def setUp(self):
        self.engine = TabFactEngine()

    def test_all_rows_literal_executes(self):
        df = pd.DataFrame({"x": ["a", "b", "c"]})
        res = self.engine.execute("eq{count{all_rows}; 3}=True", df)
        self.assertTrue(res.executable)
        self.assertTrue(res.final)

    def test_rank_with_rank_column(self):
        df = pd.DataFrame({
            "rank": ["2", "1", "3"],
            "name": ["B", "A", "C"],
        })
        # rank{all_rows; name} вернёт name у минимального rank, т.е. "A"
        res = self.engine.execute('eq{rank{all_rows; name}; "A"}=True', df)
        self.assertTrue(res.executable)
        self.assertTrue(res.final)

    def test_rank_without_rank_column_fallback_top(self):
        df = pd.DataFrame({
            "name": ["TopName", "Other"],
        })
        res = self.engine.execute('eq{rank{all_rows; name}; "TopName"}=True', df)
        self.assertTrue(res.executable)
        self.assertTrue(res.final)

    def test_aliases_work_end_to_end(self):
        df = pd.DataFrame({"x": ["2", "1"]})
        # gt -> greater (строго числовое сравнение)
        res = self.engine.execute("gt{hop{0; x}; hop{1; x}}=True", df)
        # ВАЖНО: hop{0; x} сейчас не поддерживает row=0 как int (в текущем движке row int не создаётся из Literal)
        # Поэтому этот тест ожидаемо должен быть НЕ executable.
        # Оставляем как “охраняющий” тест: если позже добавишь int-literal → станет executable.
        self.assertFalse(res.executable)

    def test_greater_non_numeric_is_not_executable(self):
        df = pd.DataFrame({"x": ["abc", "def"]})
        res = self.engine.execute("greater{hop{filter_eq{all_rows; x; abc}; x}; 1}=True", df)
        self.assertFalse(res.executable)

    def test_filter_repair_two_args_executes(self):
        df = pd.DataFrame({
            "crowd": ["100", "200", "150"],
            "name": ["a", "b", "c"],
        })
        # filter_less{all_rows; crowd, 160} -> rows with crowd < 160 : rows 0 and 2
        # count(...) == 2
        res = self.engine.execute("eq{count{filter_less{all_rows; crowd, 160}}; 2}=True", df)
        self.assertTrue(res.executable)
        self.assertTrue(res.final)


if __name__ == "__main__":
    unittest.main(verbosity=2)


usage: ipykernel_launcher.py [-h] [-v] [-q] [--locals] [--durations N] [-f]
                             [-c] [-b] [-k TESTNAMEPATTERNS]
                             [tests ...]
ipykernel_launcher.py: error: argument -f/--failfast: ignored explicit argument '/run/user/569202730/jupyter/runtime/kernel-v3012b671ef0768db572f42a2fa465ad83da5fe74d.json'


SystemExit: 2